# Stage 2: Gộp các file xlsx từ stage 1 thành 1 file csv để import vào Supabase

Notebook này đọc **trực tiếp** các file `stage1_*.xlsx` (output từ `Stage1_Gemini_Captioning`) và xuất ra **1 file CSV** import thẳng vào Supabase table `public.dataset`.

**Điểm quan trọng:**
- Chuẩn hoá đúng **15 cột** theo schema `public.dataset`.
- Các cột enum[] (`objects_present`, `auto_flags`, `error_tags`) được convert sang **Postgres array literal**: `{ "a","b" }`.
- `is_checked` chuẩn hoá về enum: `unchecked / checked / reviewed` (hỗ trợ legacy 0/1, True/False).


## 1. Cấu hình input/output

In [26]:
# !pip -q install pandas openpyxl

In [27]:
from pathlib import Path

# === CONFIG ===
INPUT_DIR = Path("../output/stage1_generated_captions")                    
PATTERN = "stage1_SGK_CanhDieu_*.xlsx"                  
OUTPUT_CSV = Path("../output/stage2_supabase_input/stage2_supabase_input.csv")

# delimiter dùng trong cell nếu Gemini ghi nhiều item trong 1 ô (vd: "person;student")
CELL_DELIMITER = ";"

# Các cột array theo schema Supabase
ARRAY_COLS = ["objects_present", "auto_flags", "error_tags"]

print("INPUT_DIR:", INPUT_DIR.resolve())
print("PATTERN:", PATTERN)
print("OUTPUT_CSV:", OUTPUT_CSV.resolve())


INPUT_DIR: D:\Drive\OneDrive - xing512\HCMUS\Year_3\01_Coursework\CSC15006_IntroNLP\02_Project_Midterm_Dataset\vn-textbook-caption-dataset\output\stage1_generated_captions
PATTERN: stage1_SGK_CanhDieu_*.xlsx
OUTPUT_CSV: D:\Drive\OneDrive - xing512\HCMUS\Year_3\01_Coursework\CSC15006_IntroNLP\02_Project_Midterm_Dataset\vn-textbook-caption-dataset\output\stage2_supabase_input\stage2_supabase_input.csv


## 2. Helper functions (chuẩn hoá schema + array literal)


In [28]:
import json
import pandas as pd

# --- Schema columns (match table public.dataset, without created_at/updated_at) ---
COLUMN_ORDER = [
    "id",
    "image",
    "page_type",
    "has_text",
    "has_table",
    "objects_present",
    "caption_short",
    "caption_detail",
    "text_in_image",
    "review_priority",
    "auto_flags",
    "notes",
    "is_checked",
    "error_tags",
    "change_log",
]

# --- Enum defaults (match supabase-schema.sql) ---
PAGE_TYPE_DEFAULT = "other"
REVIEW_PRIORITY_DEFAULT = "normal"
IS_CHECKED_DEFAULT = "unchecked"
IS_CHECKED_ALLOWED = {"unchecked", "checked", "reviewed"}


def _to_bool(v) -> bool:
    if isinstance(v, bool):
        return v
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return False
    if isinstance(v, (int, float)):
        try:
            return bool(int(v))
        except Exception:
            return False
    s = str(v).strip().lower()
    if s in ("true", "1", "yes", "y"):
        return True
    if s in ("false", "0", "no", "n", ""):
        return False
    return False


def normalize_is_checked(v) -> str:
    """Legacy support: 0/1, True/False -> unchecked/checked."""
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return IS_CHECKED_DEFAULT

    if isinstance(v, bool):
        return "checked" if v else "unchecked"

    if isinstance(v, (int, float)):
        try:
            return "checked" if int(v) != 0 else "unchecked"
        except Exception:
            return IS_CHECKED_DEFAULT

    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return IS_CHECKED_DEFAULT

    if s in ("0", "1"):
        return "checked" if s == "1" else "unchecked"

    s_norm = s.lower()
    return s_norm if s_norm in IS_CHECKED_ALLOWED else IS_CHECKED_DEFAULT


def _pg_escape_array_element(s: str) -> str:
    # escape backslash + quote for Postgres array literal
    s = s.replace("\\", "\\\\").replace('"', '\\"')
    return f'"{s}"'


def to_pg_array_literal(value, *, delimiter: str = ";", empty_literal: str = "{}") -> str:
    """Convert common spreadsheet formats into Postgres array literal text."""
    if value is None or (isinstance(value, float) and pd.isna(value)) or pd.isna(value):
        return empty_literal

    # Already list/tuple
    if isinstance(value, (list, tuple, set)):
        items = [str(x).strip() for x in value]
    else:
        s = str(value).strip()
        if s == "" or s.lower() == "nan":
            return empty_literal

        # Already Postgres array literal
        if s.startswith("{") and s.endswith("}"):
            return s

        # JSON list string
        if s.startswith("[") and s.endswith("]"):
            try:
                arr = json.loads(s)
                items = [str(x).strip() for x in arr] if isinstance(arr, list) else []
            except Exception:
                inner = s[1:-1].strip()
                items = [p.strip() for p in inner.split(",")] if inner else []
        else:
            # Delimited string (ưu tiên delimiter config, fallback comma)
            parts = s.split(delimiter) if (delimiter and delimiter in s) else s.split(",")
            items = [p.strip() for p in parts]

    # Clean: drop empties, de-dup preserving order
    cleaned = []
    seen = set()
    for it in items:
        if not it or it.lower() == "nan":
            continue
        if it not in seen:
            cleaned.append(it)
            seen.add(it)

    if not cleaned:
        return empty_literal

    return "{" + ",".join(_pg_escape_array_element(it) for it in cleaned) + "}"


def ensure_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    for c in COLUMN_ORDER:
        if c not in df.columns:
            if c in ("has_text", "has_table"):
                df[c] = False
            elif c == "page_type":
                df[c] = PAGE_TYPE_DEFAULT
            elif c == "review_priority":
                df[c] = REVIEW_PRIORITY_DEFAULT
            elif c == "is_checked":
                df[c] = IS_CHECKED_DEFAULT
            else:
                df[c] = ""

    df["has_text"] = df["has_text"].apply(_to_bool)
    df["has_table"] = df["has_table"].apply(_to_bool)
    df["is_checked"] = df["is_checked"].apply(normalize_is_checked)

    return df[COLUMN_ORDER]


def read_xlsx_all_sheets(path: Path) -> pd.DataFrame:
    """Đọc tất cả sheet rồi concat (phòng file có nhiều sheet)."""
    sheets = pd.read_excel(path, sheet_name=None)
    frames = [d for d in sheets.values() if d is not None and not d.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=COLUMN_ORDER)


## 3. Merge tất cả XLSX thành 1 DataFrame


In [29]:
files = sorted([p for p in INPUT_DIR.glob(PATTERN) if p.is_file()])
if not files:
    raise FileNotFoundError(f"No xlsx matched '{PATTERN}' in {INPUT_DIR.resolve()}")

frames = []
total_rows = 0

for i, f in enumerate(files, 1):
    print(f"[{i}/{len(files)}] reading {f.name} ...")
    df = read_xlsx_all_sheets(f)
    if df.empty:
        print("  ⚠ empty -> skipped")
        continue
    df = ensure_columns(df)

    # convert array cols to Postgres array literal
    for col in ARRAY_COLS:
        if col in df.columns:
            df[col] = df[col].apply(lambda v: to_pg_array_literal(v, delimiter=CELL_DELIMITER))
            df[col] = df[col].replace({"": "{}"}).fillna("{}")

    frames.append(df)
    total_rows += len(df)
    print(f"  ✓ rows={len(df)}")

if not frames:
    raise ValueError("All inputs were empty. Nothing to export.")

merged = pd.concat(frames, ignore_index=True)
print("="*80)
print("Total merged rows:", total_rows)
print("Columns:", list(merged.columns))
print("="*80)
merged.head(3)


[1/10] reading stage1_SGK_CanhDieu_DaoDuc_1.xlsx ...
  ✓ rows=5
[2/10] reading stage1_SGK_CanhDieu_DaoDuc_2.xlsx ...
  ✓ rows=5
[3/10] reading stage1_SGK_CanhDieu_DaoDuc_3.xlsx ...
  ✓ rows=5
[4/10] reading stage1_SGK_CanhDieu_TiengViet_2_Tap1.xlsx ...
  ✓ rows=5
[5/10] reading stage1_SGK_CanhDieu_TiengViet_2_Tap2.xlsx ...
  ✓ rows=5
[6/10] reading stage1_SGK_CanhDieu_Toan_1.xlsx ...
  ✓ rows=5
[7/10] reading stage1_SGK_CanhDieu_Toan_3_Tap1.xlsx ...
  ✓ rows=5
[8/10] reading stage1_SGK_CanhDieu_TuNhienVaXaHoi_1.xlsx ...
  ✓ rows=5
[9/10] reading stage1_SGK_CanhDieu_TuNhienVaXaHoi_2.xlsx ...
  ✓ rows=5
[10/10] reading stage1_SGK_CanhDieu_TuNhienVaXaHoi_3.xlsx ...
  ✓ rows=5
Total merged rows: 50
Columns: ['id', 'image', 'page_type', 'has_text', 'has_table', 'objects_present', 'caption_short', 'caption_detail', 'text_in_image', 'review_priority', 'auto_flags', 'notes', 'is_checked', 'error_tags', 'change_log']


,id,image,page_type,has_text,has_table,objects_present,caption_short,caption_detail,text_in_image,review_priority,auto_flags,notes,is_checked,error_tags,change_log
0,SGK_CanhDieu_DaoDuc_1_page_001,SGK_CanhDieu_DaoDuc_1_page_001.png,cover,True,False,"{""person_child"",""student"",""table"",""pencil"",""il...","Bìa sách giáo khoa Đạo đức 1, bộ Cánh Diều. Ản...",Góc trên bên trái là logo với hình ảnh cánh di...,Cánh Diều\nLƯU THU THUỶ (Tổng Chủ biên kiêm Ch...,normal,{},NaN,unchecked,{},NaN
1,SGK_CanhDieu_DaoDuc_1_page_002,SGK_CanhDieu_DaoDuc_1_page_002.png,blank,False,False,{},"Trang trắng, không có nội dung văn bản hay hìn...","Toàn bộ trang là một nền trắng, không xuất hiệ...",NaN,low,"{""POSSIBLE_BLANK""}",Ảnh hoàn toàn trắng.,unchecked,{},NaN
2,SGK_CanhDieu_DaoDuc_1_page_003,SGK_CanhDieu_DaoDuc_1_page_003.png,title_page,True,False,{},Trang tiêu đề sách giáo khoa Đạo đức 1 (bộ Cán...,Đây là trang tiêu đề bên trong sách. Ở phía tr...,LƯU THU THỦY (Tổng Chủ biên kiêm Chủ biên) - N...,low,{},NaN,unchecked,{},NaN


## 4. Quick sanity checks
Các check kiểm tra xem CSV có bị lỗi trước khi import Supabase.


In [30]:
# 4.1) thiếu id/image
missing_id = merged["id"].astype(str).str.strip().eq("").sum()
missing_image = merged["image"].astype(str).str.strip().eq("").sum()
print("missing id:", missing_id)
print("missing image:", missing_image)

# 4.2) distribution
print("\n-- is_checked distribution --")
print(merged["is_checked"].value_counts(dropna=False))

print("\n-- page_type distribution --")
print(merged["page_type"].value_counts(dropna=False).head(20))

# 4.3) check array literal format
def looks_like_pg_array(s):
    s = str(s)
    return s.startswith("{") and s.endswith("}")

for col in ARRAY_COLS:
    bad = (~merged[col].apply(looks_like_pg_array)).sum() if col in merged.columns else 0
    print(f"bad array literal in {col}: {bad}")


missing id: 0
missing image: 0

-- is_checked distribution --
is_checked
unchecked    50
Name: count, dtype: int64

-- page_type distribution --
page_type
title_page    14
cover         11
other          9
content        6
blank          5
toc            4
exercise       1
Name: count, dtype: int64
bad array literal in objects_present: 0
bad array literal in auto_flags: 0
bad array literal in error_tags: 0


## 5. Export CSV (Supabase import)
CSV sẽ được ghi với encoding `utf-8-sig` để mở bằng Excel/VN locale không lỗi dấu.


In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUTPUT_CSV.resolve())

Saved: D:\Drive\OneDrive - xing512\HCMUS\Year_3\01_Coursework\CSC15006_IntroNLP\02_Project_Midterm_Dataset\vn-textbook-caption-dataset\output\stage2_supabase_input\stage2_supabase_input.csv


## 6. Notes khi import vào Supabase

- Import vào table **`public.dataset`**.
- Nếu Supabase import báo lỗi enum/array:
  - Kiểm tra các giá trị trong `page_type`, `review_priority`, `is_checked` có đúng enum không.
  - Các cột `objects_present`, `auto_flags`, `error_tags` phải là `{}` hoặc `{ "..." }`.
- Nếu `id` trùng nhau, Supabase sẽ báo conflict (tuỳ bạn import kiểu insert hay upsert).
